#  Computation of BOP Features

- input: atoms objects
- output here: all the descriptors comming from BOP (averages, atomic site based, etc)

#  Calculation of features from available libraries

- input curated BS

# TODO
- [X] Calculate site specific averages
- [ ] Distributions of descriptors / characterization of descriptors

In [1]:
import os
import sys
sys.path.insert(0, '/home/storage/fortimtb/CuadernoTrabajo/bopfoxfeaturizer/')
from BopFoxFeaturizer.Featurizer import Featurizer, BopfoxFeatures
import pandas as pd

## options:

In [2]:
dataset = 'Cr-Co-W'
atomsobjectloc = os.path.join(dataset, 'Atomsobjects')
components = dataset.replace('-','')
models = ['canonical','projections', 'projections_os']
cutoff = 'table'
atoms = 'initial'
moments = 16

In [3]:
# only loop over the models
atomspickle =  os.path.join(atomsobjectloc, f'{components}-sorted-POSCAR-{atoms}-rescaled-AtomsObjects.pkl')
AtomsObjects = pd.read_pickle(atomspickle).dropna(how='any')

In [4]:
results = {}
resultspickle = {}
for model in models:
    modelsfile = os.path.join('models', f'{components}_{model}.bx')
    print('atoms: ', atoms, 'model: ', model, '  cutoff: ', cutoff, ' moments:', moments)
    resultspickle[model] = os.path.join(dataset, 'Descriptors', f'{components}_{atoms}_{model}_{cutoff}_WUBIND_{moments}.pkl')
    BOPC = BopfoxFeatures(
            AtomsObjects['atoms'],modelsfile, modelname=model,
            cutoffby=cutoff, 
            binary = '/home/storage/fortimtb/CuadernoTrabajo/oldrepobopfox/src/bopfox_mpi',
            moments = moments
            )
    BOPC.calculate_bop_forall(ForceKeepSpecies=True,
            input_pickle = resultspickle[model]
            )
    results[model] = BOPC.RESULTS

atoms:  initial model:  canonical   cutoff:  table  moments: 16


  0%|          | 0/1681 [00:00<?, ?it/s] 

atoms:  initial model:  projections   cutoff:  table  moments: 16


  0%|          | 0/1681 [00:00<?, ?it/s] 

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



return None
return None
return None
return None
return None
return None
return None
return None
return None
return None
return None
return None
return None
return None
return None
return None
return None


# Failed BOP Calculations 

In [5]:
for model, result in results.items():
    print(f'{model} model:')
    print(result.isna().sum())
    print(result[result.isna().any(axis=1)])
    print('=================')

canonical model:
U_bind              0
U_bond_atom         0
U_bond_atom_list    0
moments             0
normed_moments      0
an                  0
bn                  0
sigma               0
Ainf                0
Binf                0
dtype: int64
Empty DataFrame
Columns: [U_bind, U_bond_atom, U_bond_atom_list, moments, normed_moments, an, bn, sigma, Ainf, Binf]
Index: []
projections model:
U_bind              0
U_bond_atom         0
U_bond_atom_list    0
moments             0
normed_moments      0
an                  0
bn                  0
sigma               0
Ainf                0
Binf                0
dtype: int64
Empty DataFrame
Columns: [U_bind, U_bond_atom, U_bond_atom_list, moments, normed_moments, an, bn, sigma, Ainf, Binf]
Index: []
projections_os model:
U_bind               0
U_bond_atom          0
U_bond_atom_list     0
moments             18
normed_moments       0
an                   0
bn                   0
sigma                0
Ainf                 0
Binf           

##  remove bad data

In [6]:
removenans = [result.dropna(inplace=True) for model, result in results.items()]

In [7]:
for model, result in results.items():
    print(f'{model} model:')
    print(result.isna().sum())
    print(result[result.isna().any(axis=1)])
    print('=================')

canonical model:
U_bind              0
U_bond_atom         0
U_bond_atom_list    0
moments             0
normed_moments      0
an                  0
bn                  0
sigma               0
Ainf                0
Binf                0
dtype: int64
Empty DataFrame
Columns: [U_bind, U_bond_atom, U_bond_atom_list, moments, normed_moments, an, bn, sigma, Ainf, Binf]
Index: []
projections model:
U_bind              0
U_bond_atom         0
U_bond_atom_list    0
moments             0
normed_moments      0
an                  0
bn                  0
sigma               0
Ainf                0
Binf                0
dtype: int64
Empty DataFrame
Columns: [U_bind, U_bond_atom, U_bond_atom_list, moments, normed_moments, an, bn, sigma, Ainf, Binf]
Index: []
projections_os model:
U_bind              0
U_bond_atom         0
U_bond_atom_list    0
moments             0
normed_moments      0
an                  0
bn                  0
sigma               0
Ainf                0
Binf                0
dt

In [ ]:
for (model, resultpckl), result in zip(resultspickle.items(), results.values()):
    result.to_pickle(resultpckl)

# Arrangement and averaging BOP Features 

In [8]:
from Tools.DatasetTools import GeneralFeaturizer as gf

In [9]:
descriptorlocation = os.path.join(dataset, 'Descriptors' )

In [10]:
PyscalFeaturesFile = os.path.join(descriptorlocation, 'pyscal_steinhardt.kpl')

In [11]:
PyscalFeatures = pd.read_pickle(PyscalFeaturesFile)

In [12]:
from importlib.machinery import SourceFileLoader
gf = SourceFileLoader('gf', 'Tools/DatasetTools/GeneralFeaturizer.py').load_module()

In [13]:
results['projections_os'].loc['Co_pv1.bcc.NM']

U_bind                                                      -3.241558
U_bond_atom                                                 -3.241558
U_bond_atom_list                                      [-3.2415577854]
moments             [[-7.9, 63.8731, -528.124, 4459.43, -38384.5, ...
normed_moments      [[-0.4482412313, 0.2064162918, -0.0975457428, ...
an                  [[-7.9, -8.17995, -7.99871, -8.57733, -8.58451...
bn                  [[0.0, 1.20959, 0.9623009999999999, 1.44373, 1...
sigma               [[1.0, -0.8952233869, -0.1736864879, 1.0055890...
Ainf                                                  [-0.5865655839]
Binf                                                   [8.1579224584]
Name: Co_pv1.bcc.NM, dtype: object

In [14]:
results['projections'].loc['Co_pv1.bcc.NM']

U_bind                                                      -1.250241
U_bond_atom                                                 -1.250241
U_bond_atom_list                                      [-1.2502412713]
moments             [[0.0, 0.125151, -0.00117081, 0.0234185, -0.00...
normed_moments      [[0.0165566172, 0.237518144, 0.0087326067, 0.0...
an                  [[0.0, -0.00935514, 0.00674691, -0.0162305, -0...
bn                  [[0.0, 0.353767, 0.248763, 0.319504, 0.318505,...
sigma               [[1.0, 0.0330667282, -0.0497417457, 0.00360922...
Ainf                                                  [-0.0120251727]
Binf                                                   [0.3631530696]
Name: Co_pv1.bcc.NM, dtype: object

In [22]:
resultscnav = {}
for model, result in results.items():
    columnstoexpand = result.columns.drop(['U_bind','U_bond_atom', 'U_bond_atom_list'])
    df = gf.array_expansions(result, columnstoexpand)
    ThisCoordination = PyscalFeatures.pyscal_cn[result.index]
    resultscnav[model] = gf.featurize_dataframe(df, ThisCoordination)
    averaged_bop_file = os.path.join(descriptorlocation,'CNAV_'+os.path.basename(resultspickle[model]))
    resultscnav[model].to_pickle(averaged_bop_file)

canonical
projections
projections_os
